# LogiTrack Shipment Data Audit

## Part A â€” Profile and Audit the Shipment Dataset

Prepared for: VP of Operations, LogiTrack Global Logistics
Date: June 2026

In [ ]:
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

df_raw = pd.read_csv('logitrack_raw.csv')
print(f"Dataset shape: {df_raw.shape}")
df_raw.head(3)

In [ ]:
profile = ProfileReport(df_raw, title='LogiTrack Shipment Audit', minimal=True)
profile.to_file('logitrack_profile.html')
print("Profile report exported to logitrack_profile.html")

In [ ]:
# Q1: Missing shipment_id
missing_sid = df_raw['shipment_id'].isna().sum()
print(f"Q1: {missing_sid} shipments are missing shipment_id")
print()
print("Can these rows be used in model training?")
print("NO â€” Model training requires a unique identifier to join predictions back to the original")
print("shipment record. Without a shipment_id, we cannot associate a prediction with a specific")
print("shipment, nor can we validate the model's performance against real outcomes.")
print()
print("Can they be used in production scoring?")
print("NO â€” In production, the entire purpose is to notify a customer about a specific shipment.")
print("Without a shipment_id, the customer success team cannot identify which client to call,")
print("and the notification cannot be delivered. Production scoring is useless without the ID.")

In [ ]:
# Q2: Negative weight
neg_weight = (df_raw['weight_kg'] < 0).sum()
print(f"Q2: {neg_weight} rows have negative weight_kg")
print()
print("Likely source: In a logistics system, negative weight values almost always come from")
print("manual data-entry error at the warehouse â€” an operator accidentally types a minus sign")
print("before the weight, or a scale interface sends a negative calibration value. It can also")
print("occur when an EDI feed misinterprets a weight field's sign bit.")

In [ ]:
# Q3: Missing delayed_flag
missing_df = df_raw['delayed_flag'].isna().sum()
print(f"Q3: {missing_df} rows are missing delayed_flag")
print()
print("Why this field makes or breaks the AI use case:")
print("delayed_flag is the target variable (label) for supervised learning. Without it, the model")
print("cannot learn the pattern between shipment features and actual delay outcomes. This is a")
print("classification problem â€” if we don't have labels, we cannot train a classifier.")

In [ ]:
# Q4: Carrier with highest average delay
clean = df_raw.dropna(subset=['shipment_id', 'delayed_flag'])
clean = clean[clean['weight_kg'] > 0]
avg_delay = clean.groupby('carrier')['delay_days'].mean().sort_values(ascending=False)
print("Q4: Average delay_days by carrier")
print(avg_delay)
print(f"\nCarrier with highest average delay: {avg_delay.index[0]} ({avg_delay.iloc[0]:.2f} days)")

In [ ]:
# Q5: Customs status most associated with delays
ct = pd.crosstab(clean['customs_status'], clean['delayed_flag'])
ct['delay_rate'] = ct[1] / (ct[0] + ct[1])
print("Q5: Customs status vs delayed_flag")
print(ct)
print(f"\nCustoms status most associated with delays: {ct['delay_rate'].idxmax()} ({ct['delay_rate'].max():.1%})")

## Q6 â€” Sentence for VP to Present to the Board

> "Our analysis of 350 shipment records found that 20% contain missing or invalid data critical for AI training â€” including 30 shipments without a tracking ID and 25 records missing delay status. We recommend a one-week data remediation phase to fix these entry workflows before the AI system can go live."

This sentence is designed for the VP of Operations to present to the Board in under 30 seconds. It quantifies the problem (20%), gives concrete examples (30 IDs, 25 statuses), and proposes a specific, short timeline (1 week) rather than an open-ended delay.